<a href="https://colab.research.google.com/github/idialloaka-ai/DAILYCHALLENGE/blob/master/daylichallenge_W8D2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from typing import Annotated, Literal
from typing_extensions import TypedDict
from collections.abc import Iterable
from random import randint
from pprint import pprint

from IPython.display import Image, display

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

!pip install langchain-google-genai
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_core.messages.ai import AIMessage
from langchain_core.messages.tool import ToolMessage

# --- Use Colab Secrets Manager for your API key ---
from google.colab import userdata
GOOGLE_API_KEY = userdata.get('sk-a3d1596a71394d4f9dd52e7107bdf795') # Make sure you've added your API key to Colab Secrets
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

print(" Imports OK, clé configurée.")

In [ ]:
class OrderState(TypedDict):
    messages: Annotated[list, add_messages]
    order: list[str]
    finished: bool

BARISTABOT_SYSINT = (
    "system",
    "You are a BaristaBot, an interactive cafe ordering system. A human will talk to you about the "
    "available products you have and you will answer any questions about menu items (and only about "
    "menu items - no off-topic discussion, but you can chat about the products and their history). "
    "The customer will place an order for 1 or more items from the menu, which you will structure "
    "and send to the ordering system after confirming the order with the human. "
    "\n\n"
    "Add items to the customer's order with add_to_order, and reset the order with clear_order. "
    "To see the contents of the order so far, call get_order (this is shown to you, not the user). "
    "Always confirm_order with the user (double-check) before calling place_order. Calling confirm_order will "
    "display the order items to the user and returns their response to seeing the list. Their response may contain modifications. "
    "Always verify and respond with drink and modifier names from the MENU before adding them to the order. "
    "If you are unsure a drink or modifier matches those on the MENU, ask a question to clarify or redirect. "
    "You only have the modifiers listed on the menu. "
    "Once the customer has finished ordering items, call confirm_order to ensure it is correct then make "
    "any necessary updates and then call place_order. Once place_order has returned, thank the user and "
    "say goodbye!",
)

WELCOME_MSG = "Welcome to the BaristaBot cafe. Type `q` to quit. How may I serve you today?"
print(" Prompt système défini.")

In [ ]:
# Modèle valide : 'gemini-2.5-flash' ou 'gemini-2.0-flash'
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

def chatbot(state: OrderState) -> OrderState:
    message_history = [BARISTABOT_SYSINT] + state["messages"]
    return {"messages": [llm.invoke(message_history)]}

graph_builder = StateGraph(OrderState)
graph_builder.add_node("chatbot", chatbot)
graph_builder.add_edge(START, "chatbot")
chat_graph = graph_builder.compile()
display(Image(chat_graph.get_graph().draw_mermaid_png()))
print("Premier graphe construit.")

In [ ]:
user_msg = "Hello, what can you do?"
state = chat_graph.invoke({"messages": [("user", user_msg)]})
print("--- Résultat premier tour ---")
for msg in state["messages"]:
    print(f"{type(msg).__name__}: {msg.content}")

user_msg = "That's cool. What is your name?"
state["messages"].append(("user", user_msg))
state = chat_graph.invoke(state)
print("\n--- Résultat deuxième tour ---")
for msg in state["messages"]:
    print(f"{type(msg).__name__}: {msg.content}")

In [ ]:
def human_node(state: OrderState) -> OrderState:
    last_msg = state["messages"][-1]
    print("Model:", last_msg.content)
    user_input = input("User: ")
    if user_input.lower() in {"q", "quit", "exit", "goodbye"}:
        state["finished"] = True
    return state | {"messages": [("user", user_input)]}

def chatbot_with_welcome_msg(state: OrderState) -> OrderState:
    if state["messages"]:
        new_output = llm.invoke([BARISTABOT_SYSINT] + state["messages"])
    else:
        new_output = AIMessage(content=WELCOME_MSG)
    return state | {"messages": [new_output]}

graph_builder = StateGraph(OrderState)
graph_builder.add_node("chatbot", chatbot_with_welcome_msg)
graph_builder.add_node("human", human_node)
graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot", "human")
chat_with_human_graph = graph_builder.compile()
display(Image(chat_with_human_graph.get_graph().draw_mermaid_png()))
print(" Graphe avec humain construit.")

In [ ]:
def maybe_exit_human_node(state: OrderState) -> Literal["chatbot", "__end__"]:
    if state.get("finished", False):
        return END
    else:
        return "chatbot"

graph_builder = StateGraph(OrderState)
graph_builder.add_node("chatbot", chatbot_with_welcome_msg)
graph_builder.add_node("human", human_node)
graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot", "human")
graph_builder.add_conditional_edges("human", maybe_exit_human_node)
chat_with_human_graph = graph_builder.compile()
display(Image(chat_with_human_graph.get_graph().draw_mermaid_png()))
print(" Sortie conditionnelle ajoutée.")

In [ ]:
# CELLULE 8 : MENU AVEC OUTIL (CORRIGÉ)

@tool(description="Provides the latest up-to-date menu with drinks, modifiers, and availability.")
def get_menu() -> str:
    """Return the full cafe menu."""
    return """
    MENU:
    Coffee Drinks:
    Espresso
    Americano
    Cold Brew

    Coffee Drinks with Milk:
    Latte
    Cappuccino
    Cortado
    Macchiato
    Mocha
    Flat White

    Tea Drinks:
    English Breakfast Tea
    Green Tea
    Earl Grey

    Tea Drinks with Milk:
    Chai Latte
    Matcha Latte
    London Fog

    Other Drinks:
    Steamer
    Hot Chocolate

Modifiers:
    Milk options: Whole, 2%, Oat, Almond, 2% Lactose Free; Default option: whole
    Espresso shots: Single, Double, Triple, Quadruple; default: Double
    Caffeine: Decaf, Regular; default: Regular
    Hot-Iced: Hot, Iced; Default: Hot
    Sweeteners (option to add one or more): vanilla sweetener, hazelnut sweetener, caramel sauce, chocolate sauce, sugar free vanilla sweetener
    Special requests: any reasonable modification that does not involve items not on the menu, for example: 'extra hot', 'one pump', 'half caff', 'extra foam', etc.

    "dirty" means add a shot of espresso to a drink that doesn't usually have it, like "Dirty Chai Latte".
    "Regular milk" is the same as 'whole milk'.
    "Sweetened" means add some regular sugar, not a sweetener.

    Soy milk has run out of stock today, so soy is not available.
  """

tools = [get_menu]
tool_node = ToolNode(tools)
llm_with_tools = llm.bind_tools(tools)

def maybe_route_to_tools(state: OrderState) -> Literal["tools", "human"]:
    if not (msgs := state.get("messages", [])):
        raise ValueError("No messages found when parsing state.")
    msg = msgs[-1]
    if hasattr(msg, "tool_calls") and len(msg.tool_calls) > 0:
        return "tools"
    else:
        return "human"

def chatbot_with_tools(state: OrderState) -> OrderState:
    defaults = {"order": [], "finished": False}
    if state["messages"]:
        new_output = llm_with_tools.invoke([BARISTABOT_SYSINT] + state["messages"])
    else:
        new_output = AIMessage(content=WELCOME_MSG)
    return defaults | state | {"messages": [new_output]}

graph_builder = StateGraph(OrderState)
graph_builder.add_node("chatbot", chatbot_with_tools)
graph_builder.add_node("human", human_node)
graph_builder.add_node("tools", tool_node)
graph_builder.add_conditional_edges("chatbot", maybe_route_to_tools)
graph_builder.add_conditional_edges("human", maybe_exit_human_node)
graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge(START, "chatbot")
graph_with_menu = graph_builder.compile()

display(Image(graph_with_menu.get_graph().draw_mermaid_png()))
print(" Graphe avec menu et outils construit.")

In [ ]:
# CELLULE 9 : OUTILS DE COMMANDE (CORRIGÉS)

# Ces fonctions sont vides car leur logique est dans order_node,
# mais elles doivent avoir une docstring pour que LangChain les expose au LLM.

@tool
def add_to_order(drink: str, modifiers: Iterable[str]) -> str:
    """Adds the specified drink to the customer's order, including any modifiers."""
    pass

@tool
def confirm_order() -> str:
    """Asks the customer to confirm the order and returns their response."""
    pass

@tool
def get_order() -> str:
    """Returns the current order so far, one item per line."""
    pass

@tool
def clear_order():
    """Removes all items from the customer's order."""
    pass

@tool
def place_order() -> int:
    """Sends the order to the kitchen and returns the estimated wait time in minutes."""
    pass

# Le reste de la cellule (order_node) reste identique, mais je le recopie pour vous éviter de chercher :

def order_node(state: OrderState) -> OrderState:
    """Nœud de commande. C'est ici que l'état de la commande est manipulé."""
    last_msg = state["messages"][-1]
    order = state.get("order", [])
    outbound_msgs = []
    order_placed = False

    for tool_call in last_msg.tool_calls:
        if tool_call["name"] == "add_to_order":
            drink = tool_call["args"].get("drink", "")
            modifiers = tool_call["args"].get("modifiers", [])
            modifier_str = ", ".join(modifiers) if modifiers else "no modifiers"
            order.append(f'{drink} ({modifier_str})')
            response = "\n".join(order)
        elif tool_call["name"] == "confirm_order":
            print("\n--- Confirmation de commande ---")
            print("Your order:")
            if not order:
                print("  (no items)")
            for drink in order:
                print(f"  {drink}")
            response = input("Is this correct? (yes/no) ")
        elif tool_call["name"] == "get_order":
            response = "\n".join(order) if order else "(no order)"
        elif tool_call["name"] == "clear_order":
            order.clear()
            response = "Order cleared."
        elif tool_call["name"] == "place_order":
            order_text = "\n".join(order)
            print("\n--- Envoi de la commande ---")
            print("Sending order to kitchen!")
            print(order_text)
            order_placed = True
            eta = randint(1, 5)
            response = f"Order placed! Estimated wait time: {eta} minutes."
        else:
            raise NotImplementedError(f'Unknown tool call: {tool_call["name"]}')
        outbound_msgs.append(
            ToolMessage(
                content=response,
                name=tool_call["name"],
                tool_call_id=tool_call["id"],
            )
        )
    return {"messages": outbound_msgs, "order": order, "finished": order_placed}



In [ ]:
def maybe_route_to_tools_final(state: OrderState) -> str:
    if not (msgs := state.get("messages", [])):
        raise ValueError("No messages")
    msg = msgs[-1]
    if state.get("finished", False):
        return END
    elif hasattr(msg, "tool_calls") and len(msg.tool_calls) > 0:
        tool_names = [tc["name"] for tc in msg.tool_calls]
        if any(name in tool_node.tools_by_name for name in tool_names):
            return "tools"
        else:
            return "ordering"
    else:
        return "human"

auto_tools = [get_menu]
tool_node = ToolNode(auto_tools)
order_tools = [add_to_order, confirm_order, get_order, clear_order, place_order]
llm_with_final_tools = llm.bind_tools(auto_tools + order_tools)

def chatbot_with_final_tools(state: OrderState) -> OrderState:
    defaults = {"order": [], "finished": False}
    if state["messages"]:
        new_output = llm_with_final_tools.invoke([BARISTABOT_SYSINT] + state["messages"])
    else:
        new_output = AIMessage(content=WELCOME_MSG)
    return defaults | state | {"messages": [new_output]}

graph_builder = StateGraph(OrderState)
graph_builder.add_node("chatbot", chatbot_with_final_tools)
graph_builder.add_node("human", human_node)
graph_builder.add_node("tools", tool_node)
graph_builder.add_node("ordering", order_node)

graph_builder.add_conditional_edges("chatbot", maybe_route_to_tools_final)
graph_builder.add_conditional_edges("human", maybe_exit_human_node)
graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge("ordering", "chatbot")
graph_builder.add_edge(START, "chatbot")

graph_with_order_tools = graph_builder.compile()
display(Image(graph_with_order_tools.get_graph().draw_mermaid_png()))
print(" Graphe final prêt.")

In [ ]:
.print("\n" + "="*50)
print(" BARISTABOT - Café")
print("="*50)
print("Instructions : commander, 'menu', 'q' pour quitter")
print("="*50 + "\n")

config = {"recursion_limit": 100}
try:
    final_state = graph_with_order_tools.invoke({"messages": []}, config)
    print("\n" + "="*50)
    print(" Session terminée.")
    print("="*50)
    print("\n--- État final ---")
    if final_state.get("order"):
        print("Commande passée :")
        for item in final_state["order"]:
            print(f"  - {item}")
    else:
        print("Aucune commande passée.")
    print(f"finished: {final_state.get('finished', False)}")
except Exception as e:
    print(f" Erreur : {e}")